# Réseau piéton

In [ ]:
# Geneva Cycle & Pedestrian Network Analysis
# -------------------------------------------------
# This script downloads Open data for the Canton of Geneva. 
# Only run section 5. to dowload data and save the network into segments
# -------------------------------------------------

import osmnx as ox
import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
from shapely.geometry import LineString
from shapely.ops import nearest_points
from shapely.ops import unary_union
from shapely.geometry import Point
import networkx as nx
from shapely.ops import substring


## Segmentation du réseau et export du fichier

In [ ]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'


save_path = '../../Data/output/step-1'

walk_network_name = 'shp_geneva_pedestrian_edges_all/geneva_pedestrian_priority.geojson'
bike_network_name = 'shp_geneva_cycling_edges_all/geneva_cycle_priority.geojson'

network = "walk"  # walk or bike

def fetch_network(network, operation_crs = "EPSG:2056", file_path= 'walk_network_name.geojson'):
    '''
    Fetch pedestrian or bike network from local geojson files.
    network: "walk" or "bike"
    operation_crs: CRS to project the network to (EPSG:2056 for Switzerland, projeciton in meters)
    file_path: path to the folder containing the geojson files
    returns: geopandas dataframe with the network
    '''
    if network == "walk":
        net = gpd.read_file(file_path)
        net = net.to_crs(operation_crs)
    elif network == "bike":
        net = gpd.read_file(file_path)
        net = net.to_crs(operation_crs)
    return net

In [ ]:
#import network
print("Loading pedestrian segments, replace file path -->")

net = fetch_network(network, operation_crs, file_path = f"{input_file_path}/network/network_couche_OCT/RP_final.shp")

In [ ]:
# Explode osmid lists so each row has a single osmid (works for lists, numpy arrays, and stringified lists)
import numpy as np
import ast
def to_osmid_list(x):
    if isinstance(x, (list, np.ndarray)):
        return list(x)
    if isinstance(x, str):
        try:
            val = ast.literal_eval(x)
            if isinstance(val, (list, np.ndarray)):
                return list(val)
            else:
                return [val]
        except Exception:
            return [x]
    return [x]

if "osmid" in net.columns:
    net["osmid"] = net["osmid"].apply(to_osmid_list)
    net = net.explode("osmid").reset_index(drop=True)

In [ ]:
len(net)

In [ ]:
from shapely.geometry import LineString, MultiLineString
from shapely.ops import substring
import geopandas as gpd
import pandas as pd
from tqdm import tqdm

def split_linestring(geom, segment_length):
    """Split a LineString into segments of a given length."""
    if geom.length <= segment_length:
        return [geom]
    segments = []
    start = 0.0
    while start < geom.length:
        end = min(start + segment_length, geom.length)
        seg = substring(geom, start, end)
        segments.append(seg)
        start = end
    return segments

def split_row(row, segment_length):
    geom = row.geometry
    if geom.is_empty:
        return []
    if isinstance(geom, LineString):
        segments = split_linestring(geom, segment_length)
    elif isinstance(geom, MultiLineString):
        segments = []
        for part in geom.geoms:
            segments.extend(split_linestring(part, segment_length))
    else:
        return []
    split_rows = []
    for seg in segments:
        new_row = row.copy()
        new_row.geometry = seg
        new_row["length"] = seg.length
        split_rows.append(new_row)
    return split_rows

# Desired segment length in meters
segment_length = 50

print("Splitting LineStrings into fixed-length segments...")
split_rows = []
for idx, row in tqdm(net.iterrows(), total=len(net)):
    split_rows.extend(split_row(row, segment_length))

pedestrian_segments = gpd.GeoDataFrame(split_rows, crs=net.crs)
pedestrian_segments.reset_index(drop=True, inplace=True)
# Generate segment_id as osmid + incremental number per osmid
if "osmid" in pedestrian_segments.columns:
    pedestrian_segments["segment_id"] = None
    from collections import defaultdict
    osmid_counters = defaultdict(int)
    for idx, row in pedestrian_segments.iterrows():
        osm = str(row["osmid"])
        osmid_counters[osm] += 1
        pedestrian_segments.at[idx, "segment_id"] = f"{osm}_{str(osmid_counters[osm]).zfill(3)}"
else:
    pedestrian_segments["segment_id"] = pedestrian_segments.index.astype(str).str.zfill(6)

In [ ]:
#save to gpkg file to save_path/"step1_pedestrian_segments.gpkg"
if not os.path.exists(save_path):
    os.makedirs(save_path)
output_file_gpkg = os.path.join(save_path, "step1_pedestrian_segments.gpkg")
pedestrian_segments.to_crs(target_crs).to_file(output_file_gpkg, driver="GPKG")
print(f"Pedestrian segments saved to {output_file_gpkg}")
#save to geoparquet file
output_file_parquet = os.path.join(save_path, "step1_pedestrian_segments.parquet")
pedestrian_segments.to_crs(target_crs).to_parquet(output_file_parquet)
print(f"Pedestrian segments saved to {output_file_parquet}")